# YOLOv8 문구류 3종 — Colab GPU 학습
기존 프로젝트 코드와 검증된 train/valid/test 분할을 그대로 사용합니다. 모델은 **YOLOv8n**, 100 epochs, imgsz 640, seed 42입니다. GPU가 실제로 할당된 런타임에서 위에서 아래 순서로 실행합니다.

- 기본 batch는 16이며 **CUDA 메모리 부족이 실제로 발생할 때만** 8, 4 순으로 재시도합니다.
- 로컬 PC의 일시정지한 CPU 실행과 별도의 실험입니다. 로컬 실행을 재개하거나 파일을 삭제하지 않습니다.
- Drive를 사용하면 각 epoch의 모델 저장 직후 체크포인트를 고유한 실행 폴더에 백업합니다. Drive 연결 실패 시 자동 인증 재시도를 하지 않습니다.
- 원본 영상이 아직 없으면 학습·test 평가만 진행하며 합성 영상을 제출 결과로 만들지 않습니다.
- 교수님 자료와 요구사항: 저장소의 docs/ASSIGNMENT_GUIDELINES.md. 정확한 결과보고서와 GitHub 게시 검토는 측정 결과를 받은 후 수행합니다.


## 1. 실행 설정
현재 작업용 데이터 ZIP은 Drive의 yolov8-stationery-tracking 폴더에 배치합니다. ZIP을 Colab 파일 패널에 직접 올린 경우 USE_DRIVE를 False로 설정할 수 있습니다. 이때 /content의 데이터와 결과는 런타임 종료 시 사라질 수 있으므로 결과 ZIP을 반드시 내려받습니다.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import re
import shutil
import subprocess
import sys
import uuid

if not Path("/content").is_dir():
    raise RuntimeError("This notebook must run in Google Colab, not the paused local training environment.")

REPOSITORY_URL = "https://github.com/jeongjunseo/yolov8-stationery-tracking.git"
REPO_REF = "main"  # The actual checked-out commit is recorded below.
REPO = Path("/content/yolov8-stationery-tracking")
DATA_ARCHIVE = Path("/content/stationery_dataset.zip")
USE_DRIVE = True
DRIVE_PROJECT = Path("/content/drive/MyDrive/yolov8-stationery-tracking")
SOURCE_VIDEO = Path("/content/source.mp4")  # Can also be changed to a real Drive video path.
ULTRALYTICS_VERSION = "8.4.215"
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
BACKUP_RUN = None
print("Colab run ID:", RUN_ID)


## 2. GPU 확인 및 선택적 Drive 연결
런타임의 하드웨어 가속기를 GPU로 선택한 후 실행합니다. GPU를 사용할 수 없으면 여기서 중단합니다. Drive 접근 요청은 Google의 인증 화면에서 처리합니다.


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-c",
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU is required'; "
    "print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0))"
], check=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP_RUN = DRIVE_PROJECT / "colab_runs" / RUN_ID
    BACKUP_RUN.mkdir(parents=True, exist_ok=False)
    print("Checkpoint and final-artifact backup:", BACKUP_RUN)
else:
    print("Drive backup disabled. Download the final artifact ZIP before the runtime ends.")


## 3. 기존 저장소와 의존성 준비
Colab에 설치된 CUDA PyTorch와 torchvision 버전은 유지합니다. 원본 학습 스크립트는 수정하지 않습니다. 저장소가 이미 있으면 다른 저장소인지 확인하고 기존 작업을 덮어쓰지 않습니다.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPO)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    if origin.rstrip("/") != REPOSITORY_URL:
        raise RuntimeError("Existing directory belongs to another repository; choose a new REPO path.")
    print("Preserving the existing checkout without resetting or pulling over its files.")

for required in ("scripts/train.py", "scripts/colab_support.py", "scripts/analyze_failures.py"):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f"{required} is missing; publish the prepared repository changes first.")

from importlib.metadata import version
constraints = Path("/content/stationery_colab_constraints.txt")
constraints.write_text(
    "torch==" + version("torch") + "\n"
    + "torchvision==" + version("torchvision") + "\n", encoding="utf-8"
)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt"),
    f"ultralytics=={ULTRALYTICS_VERSION}", "-c", str(constraints),
    "--disable-pip-version-check"
], check=True)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
print(pip_check.stdout, pip_check.stderr)
if pip_check.returncode:
    print("Review pip-check conflicts reported above; core GPU imports are checked separately below.")

environment_code = """
import json, platform, subprocess
from pathlib import Path
from importlib.metadata import version
import torch, ultralytics, cv2
assert torch.cuda.is_available(), "CUDA became unavailable after installation"
result = {
    "runtime": "Google Colab", "python": platform.python_version(),
    "packages": {n: version(n) for n in ("torch", "torchvision", "ultralytics", "opencv-python", "PyYAML")},
    "cuda_available": torch.cuda.is_available(), "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "device": "0",
}
Path("outputs/colab_environment.json").write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")
print(json.dumps(result, indent=2))
"""
subprocess.run([sys.executable, "-c", environment_code], cwd=REPO, check=True)
if BACKUP_RUN is not None:
    shutil.copy2(REPO / "outputs" / "colab_environment.json", BACKUP_RUN / "colab_environment.json")
SESSION = REPO / "runs" / "colab_sessions" / RUN_ID
SESSION.mkdir(parents=True, exist_ok=False)
os.chdir(REPO)
print("Checkout:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 4. 기존 데이터 복원·해시·라벨 검사
Drive의 완전한 ZIP 또는 stationery_dataset.parts.json과 분할 파일을 Colab 로컬 디스크의 ZIP으로 복사·재조립합니다. 각 part와 전체 ZIP의 SHA-256을 확인하고 Drive에는 중복 ZIP을 만들지 않습니다. 기존과 다른 데이터 파일이 있으면 덮어쓰지 않고 중단합니다. metadata 두 파일의 Windows/Unix 개행 차이만 동등하게 인정하며 원래 체크아웃 파일은 유지하고 검증 기록에 남깁니다. ZIP의 dataset_manifest.json으로 추출 파일 크기·SHA-256도 확인합니다. data/raw는 풀지 않습니다.



In [ ]:
sys.path.insert(0, str(REPO / "scripts"))
from colab_support import prepare_dataset_archive, extract_dataset
transfer = prepare_dataset_archive(DATA_ARCHIVE, DRIVE_PROJECT if USE_DRIVE else None)
print(json.dumps(transfer, indent=2))
extraction = extract_dataset(DATA_ARCHIVE, REPO)
print(json.dumps(extraction, indent=2))
subprocess.run([sys.executable, "scripts/check_dataset.py"], cwd=REPO, check=True)
counts = {
    split: len([p for p in (REPO / "data" / split / "images").iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
    for split in ("train", "valid", "test")
}
assert counts == {"train": 498, "valid": 116, "test": 73}, counts
print("Prepared split counts:", counts)
(REPO / "outputs" / "colab_dataset_check.json").write_text(
    json.dumps({"counts": counts, "transfer": transfer, "extraction": extraction, "dataset_check": "passed"}, indent=2) + "\n",
    encoding="utf-8"
)
if BACKUP_RUN is not None:
    shutil.copy2(REPO / "outputs" / "colab_dataset_check.json", BACKUP_RUN / "colab_dataset_check.json")


## 5. GPU 학습 — 100 epochs
colab_support.py는 기존 scripts/train.py의 main()을 원래 인자(device 0, batch, epochs 100)로 호출합니다. 콜백으로 Drive 백업을 수행하고 Ultralytics 8.4.215의 내부 batch 자동 변경을 차단합니다. 따라서 실제 CUDA OOM일 때만 이 노트북에서 batch 16→8→4로 재시도합니다. 모델·증강·최적화·시드 등 학습 설정은 기존 스크립트를 따릅니다. 다른 오류와 예상하지 못한 batch 값은 즉시 보고하며, 실패한 실행 폴더와 로그는 보존합니다.


In [ ]:
def run_logged(command, log_path):
    print("Running:", " ".join(map(str, command)))
    with Path(log_path).open("x", encoding="utf-8") as log:
        process = subprocess.Popen(
            list(map(str, command)), cwd=REPO, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return process.wait()

if (REPO / "models" / "best.pt").exists():
    raise FileExistsError("A completed model already exists. Preserve it and skip this cell instead of retraining.")

attempts = []
for batch in (16, 8, 4):
    command = [sys.executable, "-u", "scripts/colab_support.py", "train", "--batch", str(batch)]
    if BACKUP_RUN is not None:
        command += ["--backup-root", str(BACKUP_RUN)]
    log_path = SESSION / f"train_batch_{batch}.log"
    returncode = run_logged(command, log_path)
    attempts.append({"batch": batch, "exit_code": returncode, "log": str(log_path.relative_to(REPO))})
    (REPO / "outputs" / "colab_training_attempts.json").write_text(
        json.dumps(attempts, indent=2) + "\n", encoding="utf-8")
    if returncode == 0:
        break
    text = log_path.read_text(encoding="utf-8")
    terminal_failure = text.rsplit("Traceback (most recent call last):", 1)[-1]
    actual_cuda_oom = bool(re.search(r"CUDA out of memory|CUDA error: out of memory", terminal_failure, re.I))
    if not actual_cuda_oom or batch == 4:
        raise RuntimeError(f"Training failed (batch={batch}); inspect {log_path}. No other setting was changed.")
    print(f"Actual CUDA OOM recorded at batch {batch}; retrying the next permitted batch size.")
assert (REPO / "models" / "best.pt").is_file()
training_record = json.loads((REPO / "outputs" / "colab_training_run.json").read_text())
assert training_record["epochs_completed"] == 100
assert training_record["device"].startswith("cuda")
print("100 GPU epochs complete:", (REPO / "models" / "best.pt"))
print(json.dumps(training_record, indent=2))


## 6. 별도 test 평가 및 실패 사례 진단
첫 명령은 공식 test 지표를 저장합니다. 두 번째는 고정 confidence=0.25, 매칭 IoU=0.5로 오류 사례를 만드는 별도 진단이며 AP 지표나 임계값 튜닝을 대신하지 않습니다.


In [ ]:
if (REPO / "outputs" / "test_metrics.json").exists():
    raise FileExistsError("Existing test metrics are preserved. Inspect them rather than overwriting.")
code = run_logged([sys.executable, "-u", "scripts/evaluate.py", "--device", "0"], SESSION / "evaluate.log")
if code:
    raise RuntimeError("Evaluation failed; inspect evaluate.log.")
metrics = json.loads((REPO / "outputs" / "test_metrics.json").read_text())
print(json.dumps(metrics, ensure_ascii=False, indent=2))
code = run_logged([sys.executable, "-u", "scripts/analyze_failures.py", "--device", "0"], SESSION / "failure_analysis.log")
if code:
    raise RuntimeError("Failure-case analysis failed; inspect failure_analysis.log.")


## 7. 실제 영상이 준비된 경우에만 추적
SOURCE_VIDEO에 실제 5초 이상의 영상을 지정합니다. 영상이 없으면 이 셀은 건너뛰며 학습·평가 결과는 보존됩니다. 결과 영상의 실제 화면에서 클래스명·ID·궤적도 직접 확인해야 합니다.


In [ ]:
video = REPO / "inputs" / "source.mp4"
if SOURCE_VIDEO.is_file() and SOURCE_VIDEO.resolve() != video.resolve():
    if video.exists():
        raise FileExistsError("Existing source video is preserved; inspect paths before replacing it.")
    with SOURCE_VIDEO.open("rb") as source, video.open("xb") as target:
        shutil.copyfileobj(source, target)
if video.is_file():
    code = run_logged([
        sys.executable, "-u", "scripts/track_video.py", "--source", str(video),
        "--model", "models/best.pt", "--device", "0"
    ], SESSION / "tracking.log")
    if code:
        raise RuntimeError("Tracking failed; inspect tracking.log.")
    import cv2
    import csv
    capture = cv2.VideoCapture(str(REPO / "outputs" / "final_5s.mp4"))
    assert capture.isOpened()
    fps = capture.get(cv2.CAP_PROP_FPS)
    frames = 0
    while capture.read()[0]:
        frames += 1
    capture.release()
    assert fps > 0 and abs(frames / fps - 5.0) < 1e-6, (frames, fps)
    with (REPO / "outputs" / "trajectory.csv").open(encoding="utf-8-sig", newline="") as file:
        trajectory = list(csv.DictReader(file))
    classes = sorted({row["class_name"] for row in trajectory})
    print("Decoded video seconds:", frames / fps, "Frames:", frames, "FPS:", fps)
    print("Tracked classes:", classes, "Trajectory rows:", len(trajectory))
    if classes != ["correction_tape", "eraser", "highlighter"]:
        print("Review required: the clip did not produce tracked detections for all three classes.")
    if shutil.which("ffprobe"):
        subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=noprint_wrappers=1", "outputs/final_5s.mp4"], check=True)
else:
    print("Real source video is pending. No final video or trajectory is fabricated.")


## 8. 그래프 수집·결과 ZIP 저장
ignored runs/의 학습·평가 그래프를 report/assets/에 복사하고 모델·JSON·그림·보고서 파일을 모읍니다. 모델 성능과 실제 오류 이미지를 확인한 후 README와 보고서 내용을 최신 결과로 완성해야 합니다. ZIP 안의 진행 보고서를 그대로 최종 보고서라고 제출하지 않습니다.


In [ ]:
archive = Path("/content") / f"stationery_results_{RUN_ID}.zip"
command = [sys.executable, "scripts/colab_support.py", "collect", str(archive)]
if BACKUP_RUN is not None:
    command += ["--backup-root", str(BACKUP_RUN)]
subprocess.run(command, cwd=REPO, check=True)
print("Downloadable artifact archive:", archive)
print("Drive backup:", BACKUP_RUN if BACKUP_RUN else "disabled")
from IPython.display import FileLink, display
display(FileLink(str(archive)))
# To trigger an explicit browser download if needed:
# from google.colab import files
# files.download(str(archive))


## 중단·재연결 시
Drive 백업 폴더의 last.pt와 같은 실행의 results.csv/args.yaml을 보존합니다. 이미 완료한 실행을 다시 시작하거나 체크포인트를 덮어쓰지 않습니다. 런타임이 사라졌다면 저장된 체크포인트·데이터 경로를 확인하고 별도의 재개 절차를 사용해야 합니다. 이 노트북은 기존 로컬 CPU 체크포인트를 GPU 실험으로 몰래 이어 붙이지 않습니다.

참고: [Ultralytics 학습](https://docs.ultralytics.com/modes/train/), [추적](https://docs.ultralytics.com/modes/track/), [Colab 자원과 런타임 안내](https://research.google.com/colaboratory/faq.html).
